#  **Battery Collection CVRP: Brute Force Enumeration**

> **Optimizing E-Scooter Battery Collection Routes Through Complete Route Enumeration**

Simon Halfon & Sinai Abbou

---

##  What This Notebook Does

This notebook implements a **brute force enumeration solver** for the Capacitated Vehicle Routing Problem (CVRP) applied to e-scooter battery collection. We'll solve the problem by systematically testing *every possible route combination* to guarantee finding the optimal solution.

###  **Problem Setup**
- **Objective**: Minimize total distance traveled by collection trucks
- **Constraints**: Each scooter visited exactly once, truck capacity limits respected
- **Method**: Complete route enumeration (guaranteed optimal, but exponentially slow)



###  **Scaling Reality Check**
This approach works perfectly for **small instances** (4-6 scooters) but becomes impractical quickly:
- **4 scooters** → ~1,600 combinations → **20 milliseconds**
- **8 scooters** → ~4M combinations → **54 seconds** 
- **15 scooters** → ~62 trillion combinations → **24,000 years** 

###  **Technical Approach**
We implement the mathematical CVRP formulation using pure Python:
- **Variables**: Binary route decisions for each truck
- **Constraints**: Coverage, capacity, and flow conservation
- **Solver**: Exhaustive enumeration with feasibility checking


---
## 1: Imports and Setup
Import essential libraries for optimization, numerical computation, and visualization.

In [ ]:
import itertools
import numpy as np
import matplotlib.pyplot as plt

---
## 2: Problem Parameters
Define our small-scale CVRP instance: depot location, random scooter positions, and fleet constraints.

In [ ]:
# === User Parameters ===
np.random.seed(None)  # For reproducible results (replace by a number)

n_scooters = 4
K_trucks_idx = 2
Q_truck_capacity = 3

depot = np.array([[10, 10]])  # Fix depot at center

# Generate scooters in a square around the depot (range: [0, 20])
scooters = 10 + np.random.randint(-10, 11, size=(n_scooters, 2))

# Combine depot and scooters
coords = np.vstack([depot, scooters])


print(f"Problem Setup:")
print(f"- {n_scooters} scooters")
print(f"- {K_trucks_idx} trucks")
print(f"- Capacity: {Q_truck_capacity} scooters per truck")
print(f"- Depot at: {depot[0]}")
print(f"- Scooter locations:\n{scooters}")

---
## 3: Distance Matrix Calculation
Calculate Euclidean distances between all locations (depot and scooters) to create our cost matrix.

In [ ]:
# Compute Euclidean distance matrix between all nodes
NODES = list(range(coords.shape[0]))  # All node indices (0 = depot)
DEPOT = 0
SCOOTERS = list(range(1, coords.shape[0]))  # Indices of scooters

c = np.zeros((len(NODES), len(NODES)))
for i in NODES:
    for j in NODES:
        c[i][j] = np.linalg.norm(coords[i] - coords[j])
        
        

print("Distance matrix:")
print(c)

---
## 4: Route Generation
Generate all possible routes each truck can take. This is where the combinatorial explosion begins!

In [ ]:
# Generate all possible routes for each truck (from 1 up to capacity scooters)
all_routes = []
for r in range(1, min(Q_truck_capacity, n_scooters) + 1):
    all_routes.extend(itertools.permutations(SCOOTERS, r))

# Create route options for each truck (add depot at start/end)
truck_route_options = [
    [[DEPOT] + list(route) + [DEPOT] for route in all_routes]
    for _ in range(K_trucks_idx)
]



print(f"Generated {len(all_routes)} possible route permutations")
print(f"Each truck has {len(truck_route_options[0])} route options")
print(f"Total combinations to test: {len(truck_route_options[0])**K_trucks_idx:,}")

# Show first few routes as example
print(f"\nFirst 5 route examples:")
for i, route in enumerate(truck_route_options[0][:5]):
    print(f"  Route {i+1}: {route}")

---
## 5: Brute Force Solver
The heart of our algorithm: systematically test every route combination to find the optimal solution.

In [ ]:
best_cost = float('inf')
best_routes = None

import time
print(" Testing all route combinations...")
start_time = time.time()

# Try all combinations of routes for the trucks
total_combinations = 0
for route_combo in itertools.product(*truck_route_options):
    total_combinations += 1
    
    # Check if all scooters are covered exactly once
    scooters_visited = set()
    for route in route_combo:
        scooters_visited.update(route[1:-1])  # Exclude depot at start/end
    
    # Skip if not all scooters covered
    if len(scooters_visited) < n_scooters:
        continue
    if len(scooters_visited) > n_scooters:
        continue
    
    # Check for overlap (no scooter visited by more than one truck)
    overlap = False
    for i in range(len(route_combo)):
        for j in range(i+1, len(route_combo)):
            if set(route_combo[i][1:-1]) & set(route_combo[j][1:-1]):
                overlap = True
                break
        if overlap:
            break
    
    if overlap:
        continue
    
    # Compute total cost (sum of distances for all trucks)
    cost = sum(sum(c[route[i]][route[i+1]] for i in range(len(route)-1)) for route in route_combo)
    
    # Update best solution
    if cost < best_cost:
        best_cost = cost
        best_routes = route_combo

end_time = time.time() 
execution_time = end_time - start_time 
print(f"✅ Tested {total_combinations:,} combinations") 
print(f"⏱️ Execution time: {execution_time:.6f} seconds") 
print(f"🎯 Best solution found!")

---
## 6: Results Display
Present our optimal solution with clear metrics and verification checks.

In [ ]:
if best_routes is None:
    print(" No feasible solution found!")
else:
    print(f"\n=== OPTIMAL SOLUTION ===")
    print(f"Total Distance: {best_cost:.2f}")
    
    for i, route in enumerate(best_routes):
        scooter_count = len([node for node in route if node != 0])
        print(f"Truck {i+1}: {route} (Load: {scooter_count}/{Q_truck_capacity})")
    
    # Verify solution
    n_assigned = sum(len([node for node in route if node != 0]) for route in best_routes)
    n_used_trucks = sum(1 for route in best_routes if len([node for node in route if node != 0]) > 0)
    
    print(f"\n Solution Verification:")
    print(f"- Scooters assigned: {n_assigned}/{n_scooters}")
    print(f"- Trucks used: {n_used_trucks}/{K_trucks_idx}")
    print(f"- Average distance per scooter: {best_cost/n_scooters:.2f}")

---
# 7: Visualization
Create a beautiful plot showing the optimal routes with different colors for each truck.


In [ ]:
# Plot the solution
colors = ['red', 'blue', 'green', 'orange', 'purple']
plt.figure(figsize=(10, 8))

# Plot depot
plt.scatter(coords[0,0], coords[0,1], c='black', s=200, marker='s', 
           label='Depot', zorder=5, edgecolors='white', linewidth=2)

# Plot scooters
for idx, loc in enumerate(coords[1:]):
    plt.scatter(loc[0], loc[1], c='gray', s=100, marker='o', 
               zorder=4, edgecolors='black', linewidth=1)
    plt.text(loc[0]+0.3, loc[1]+0.3, f'S{idx+1}', fontsize=10, 
             fontweight='bold', color='black')

# Plot routes
if best_routes:
    for k, route in enumerate(best_routes):
        if len(route) > 2:  # Skip empty routes
            xs = [coords[i,0] for i in route]
            ys = [coords[i,1] for i in route]
            plt.plot(xs, ys, color=colors[k%len(colors)], linewidth=3, 
                    marker='o', markersize=8, alpha=0.7,
                    label=f'Truck {k+1} Route')

plt.title(f'Brute Force CVRP Solution\nTotal Distance: {best_cost:.2f}', 
          fontsize=14, fontweight='bold')
plt.xlabel('X Coordinate', fontsize=12)
plt.ylabel('Y Coordinate', fontsize=12)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=10)
plt.axis('equal')
plt.tight_layout()
plt.show()